<div style="background: linear-gradient(120deg, #1a3a5c 0%, #2d6a9f 60%, #4a9eda 100%); padding: 28px 36px; border-radius: 14px; display: flex; align-items: center; gap: 28px; box-shadow: 0 4px 18px rgba(0,0,0,0.18);">
    <img src='Figures/iteso.jpg' style="height: 110px; border-radius: 8px; background: white; padding: 6px; flex-shrink: 0; box-shadow: 0 2px 8px rgba(0,0,0,0.2);"/>
    <div style="border-left: 2px solid rgba(255,255,255,0.4); padding-left: 28px;">
        <h1 style="margin: 0 0 8px 0; color: white; font-size: 1.5em; line-height: 1.3;">Módulo 3: Tratamiento de Datos Faltantes</h1>
        <h3 style="margin: 0 0 8px 0; color: white; font-size: 1.15em; line-height: 1.3;">Ingeniería de Características</h3>
        <h3 style="margin: 0; color: rgba(255,255,255,0.8); font-weight: normal; font-size: 1.05em;">Maestría en Ciencia de Datos</h3>
    </div>
</div>


> Los datos faltantes no son simplemente "huecos" que rellenar con la media; son señales informativas cuyo mecanismo de generación (MCAR, MAR, MNAR) condiciona la validez estadística y el rendimiento predictivo del modelo. Una imputación inadecuada destruye la varianza, distorsiona las correlaciones y puede introducir sesgos irreparables o *data leakage*.


### Flujo de Tratamiento de datos Faltantes


```mermaid
flowchart TD
    A[Diagnóstico y Visualización de Nulidad] --> B[Identificación del Mecanismo: MCAR / MAR / MNAR]
    B --> C{¿Porcentaje de Faltantes y Patrón?}
    C -->|Muy alto / Irrelevante| D[Eliminación / Filtrado por Umbral]
    C -->|Bajo / Univariado / Rápido| E[Imputación Univariada + Missing Indicator]
    C -->|Datos Temporales / Ordenados| F[Forward/Backward Fill e Interpolación]
    C -->|Relaciones Multivariadas Fuertes| G[Modelos Predictivos: KNN / MICE / MissForest]
    D & E & F & G --> H[Validación de Distribuciones y Evaluación en Pipeline ML]
```

#### ¿Por qué los datos faltantes son un reto crítico en Ciencia de Datos?

1. **Incompatibilidad algorítmica:** La mayoría de los algoritmos de Machine Learning (`scikit-learn`, `statsmodels`, redes neuronales) no admiten matrices con `NaN`.
2. **Pérdida de potencia estadística:** La eliminación ingenua reduce el tamaño muestral efectivo ($N$), ensanchando los intervalos de confianza.
3. **Sesgo en las estimaciones:** Si los datos faltantes no son completamente aleatorios (MAR o MNAR), eliminar o imputar de forma simplista sesga los parámetros del modelo.
4. **Distorsión de la varianza y covarianza:** Asignar un valor constante a todos los faltantes concentra artificialmente la masa probabilística, reduciendo la varianza estimada y sobreestimando o alterando correlaciones.
5. **Riesgo de Data Leakage:** Imputar antes de dividir en conjuntos de entrenamiento y prueba (*train/test split*) filtra información del futuro o de evaluación hacia el entrenamiento.

## Diagnóstico y Visualización de Datos Faltantes

Antes de elegir cualquier tratamiento, el primer paso en el flujo de ingeniería de características es **diagnosticar**:
- ¿Cuántos datos faltan en cada variable (conteo y porcentaje)?
- ¿Se concentran los faltantes en observaciones específicas o en variables específicas?
- ¿Existe un patrón sistemático de ausencia o co-ocurrencia entre columnas?

### Diagnóstico tabular con `pandas`
La combinación de `.isna()`, `.sum()` y cálculos porcentuales permite construir una tabla resumen de nulidad.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd

In [ ]:
df=pd.read_csv('Data/API_SI.POV.DDAY_DS2.csv',encoding='latin-1',sep='\t')
df.head(10)

In [ ]:
df.info()

In [ ]:
# Tabla resumen de valores faltantes por columna
resumen_faltantes = pd.DataFrame({
    'Total Faltantes': df.isna().sum(),
    'Porcentaje (%)': (df.isna().mean() * 100).round(2),
    'Tipo de Dato': df.dtypes
})
resumen_faltantes = resumen_faltantes[resumen_faltantes['Total Faltantes'] > 0].sort_values(by='Porcentaje (%)', ascending=False)
print(f"Dimensiones del dataset: {df.shape[0]} filas x {df.shape[1]} columnas")
resumen_faltantes.head(15)

In [ ]:
# Visualización de la matriz booleana de faltantes con imshow
dfcopy = df.isna()

plt.figure(figsize=(10, 6))
plt.imshow(dfcopy, cmap='viridis', aspect='auto')
plt.colorbar(label='¿Es Faltante? (1 = Sí / Amarillo, 0 = No / Morado)')
plt.title('Matriz de Datos Faltantes (API_SI.POV.DDAY_DS2)', fontsize=13)
plt.xlabel('Índice de Columna')
plt.ylabel('Índice de Fila (Observación)')
plt.show()

### Visualización Especializada con `missingno`

La librería `missingno` proporciona un conjunto de herramientas gráficas diseñadas específicamente para diagnosticar el comportamiento y la correlación entre valores faltantes:

1. **`msno.bar`**: Muestra la proporción de datos presentes por variable y el conteo absoluto en el eje superior.
2. **`msno.matrix`**: Mapa visual de la matriz completa con un indicador lateral (*sparkline*) que muestra la completitud de cada fila (mínimo y máximo de columnas presentes por observación).
3. **`msno.heatmap`**: Correlación de nulidad (*Nullity Correlation*) que mide la dependencia entre faltantes:
   - $+1$: Si la variable $A$ es faltante, la variable $B$ casi con certeza también es faltante.
   - $-1$: Si la variable $A$ es faltante, la variable $B$ casi con certeza está presente.
   - $0$: No existe correlación entre las ausencias.
4. **`msno.dendrogram`**: Agrupamiento jerárquico (*hierarchical clustering*) que revela grupos de variables con patrones de ausencia similares.

In [ ]:
# Instalación de missingno si es necesario
#!pip install missingno

In [ ]:
# Visualización de datos faltantes con missingno
import missingno as msno

In [ ]:
titanic_df = sns.load_dataset('titanic')
titanic_df

In [ ]:
#Bar plot 
msno.bar(titanic_df, color='steelblue')
#color='steelblue')
#color="tomato")
#color="tab:green")

In [ ]:
msno.matrix(titanic_df)

In [ ]:
# Correlación de nulidad (Nullity Correlation Heatmap)
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

msno.heatmap(titanic_df, ax=axes[0], fontsize=10)
axes[0].set_title('Correlación de Nulidad (msno.heatmap)', fontsize=12)

msno.dendrogram(titanic_df, ax=axes[1], orientation='top', fontsize=10)
axes[1].set_title('Dendrograma de Patrones de Nulidad (msno.dendrogram)', fontsize=12)

plt.tight_layout()
plt.show()

##  Clasificación de Datos Faltantes (Taxonomía de Rubin, 1976)

Donald Rubin (1976) formalizó matemáticamente los mecanismos generadores de datos faltantes.

Sea $Y = (Y_{\text{obs}}, Y_{\text{mis}})$ la matriz de datos completa, donde $Y_{\text{obs}}$ representa los valores observados y $Y_{\text{mis}}$ los valores ausentes. Definimos la **matriz indicadora de ausencia** $M$ con elementos:

$$
M_{ij} = \begin{cases} 1 & \text{si } Y_{ij} \text{ es faltante (NaN)} \\ 0 & \text{si } Y_{ij} \text{ es observado} \end{cases}
$$

El comportamiento de $M$ define los tres mecanismos fundamentales:

```
                  ┌─────────────────────────────────────────────────────────┐
                  │                 MECANISMOS DE PÉRDIDA                   │
                  └─────────────────────────────────────────────────────────┘
                                               │
             ┌─────────────────────────────────┼─────────────────────────────────┐
             ▼                                 ▼                                 ▼
       ┌───────────┐                     ┌───────────┐                     ┌───────────┐
       │   MCAR    │                     │    MAR    │                     │   MNAR    │
       │ Complot.  │                     │  Al Azar  │                     │  No al    │
       │  al Azar  │                     │           │                     │   Azar    │
       └─────┬─────┘                     └─────┬─────┘                     └─────┬─────┘
             │                                 │                                 │
   P(M|Y) = P(M)                      P(M|Y) = P(M|Y_obs)               P(M|Y) = P(M|Y_obs, Y_mis)
             │                                 │                                 │
    No depende de nada                Depende solo de lo                Depende del propio valor
      observado ni no                 observado (ej. edad               no observado (ej. salario
        observado                      predice ingreso)                  alto se oculta)
```

<img src="Figures/tipo_faltante.png" width="550" style="display: block; margin: auto; border-radius: 8px;">

### 1. MCAR (Missing Completely at Random)

$$P(M \mid Y_{\text{obs}}, Y_{\text{mis}}, \phi) = P(M \mid \phi)$$

**Definición:** La probabilidad de que un valor falte es totalmente independiente de los valores observados y de los valores ausentes. Los datos observados son una submuestra aleatoria simple de la población original.

**Propiedades clave:**
- Eliminar filas incompletas (*listwise deletion*) **no introduce sesgo** en las medias ni en los coeficientes de regresión, pero reduce la potencia estadística ($N$ menor).
- La imputación por media/mediana preserva la media muestral, pero contrae la varianza.

**Ejemplos en la vida real:**
1. Un tubo de ensayo se rompe accidentalmente en el laboratorio.
2. Pérdida aleatoria de paquetes de red en un sensor IoT.
3. Un encuestado salta una página entera por descuido físico.

In [ ]:
# Ejemplo MCAR: Simulación
import numpy as np
import pandas as pd

np.random.seed(42)
data = pd.DataFrame({
    'edad': np.random.randint(20, 60, 100),
    'ingreso': np.random.randint(20000, 80000, 100)
})
# Introducimos valores faltantes aleatoriamente (MCAR)
mask = np.random.rand(*data.shape) < 0.1
data_mcar = data.mask(mask)
print('Datos con valores faltantes MCAR:')
print(data_mcar.head())

#### Diagnóstico Práctico de MCAR vs. MAR (Prueba de diferencias de medias)

La **Prueba de Little (1988)** evalúa globalmente la hipótesis nula:
- $H_0$: Los datos son MCAR (los patrones de faltantes comparten los mismos parámetros poblacionales).
- $H_1$: Los datos NO son MCAR (son MAR o MNAR).

**Diagnóstico univariado directo:**
Podemos verificar si la ausencia de una variable $Y$ está asociada con otra variable observada $X$ mediante una prueba $t$ de Student o Mann-Whitney: comparamos la media de $X$ en el grupo con $Y$ observado vs. el grupo con $Y$ faltante. Si hay diferencia estadísticamente significativa ($p < 0.05$), rechazamos MCAR en favor de MAR.

### 2. MAR (Missing At Random)

$$P(M \mid Y_{\text{obs}}, Y_{\text{mis}}, \phi) = P(M \mid Y_{\text{obs}}, \phi)$$

**Definición:** La probabilidad de que un dato esté ausente depende de otras variables observadas en el dataset, pero condicional a esas variables, la ausencia no depende del valor que falta.

> **¡Ojo con el nombre!** A pesar de llamarse "aleatorio", **no es aleatorio simple**. Es aleatorio *condicional* a las variables observadas.

**Propiedades clave:**
- La eliminación de casos completos (*listwise deletion*) **introduce sesgo**.
- Métodos condicionales y multivariados (regresión, KNN, MICE, MissForest) son **válidos e insesgados** porque aprovechan la información de las variables predictoras correlacionadas.

**Ejemplos:**
1. En encuestas de salud, los pacientes de mayor edad omiten con mayor frecuencia preguntas sobre ingresos económicos; pero dentro del mismo grupo de edad, la omisión es aleatoria.
2. En el dataset *Titanic*, la probabilidad de que falte `Age` es mucho mayor para pasajeros de 3ra clase (`Pclass = 3`) que para los de 1ra clase.

In [ ]:
# Ejemplo MAR: Simulación
data_mar = data.copy()
# Si la edad es mayor a 50, hay más probabilidad de que ingreso sea NaN
prob = np.where(data_mar['edad'] > 50, 0.4, 0.05)
mask = np.random.rand(len(data_mar)) < prob
data_mar.loc[mask, 'ingreso'] = np.nan
print('Datos con valores faltantes MAR:')
print(data_mar.head(10))

### 3. MNAR (Missing Not At Random / Non-Ignorable)

$$P(M \mid Y_{\text{obs}}, Y_{\text{mis}}, \phi) \neq P(M \mid Y_{\text{obs}}, \phi)$$

**Definición:** La probabilidad de que un valor falte depende directamente del propio valor no observado, incluso tras controlar por todas las variables observadas disponibles.

**Consecuencias y Estrategias en Feature Engineering:**
- Es el escenario más desafiante: los métodos estándar (media, mediana, KNN, MICE) resultan **sesgados**.
- La ausencia en sí misma constituye una variable latente de alto poder predictivo.
- **Estrategia fundamental:** Crear un **Indicador de Ausencia** (*Missing Indicator*) como característica binaria explícita ($I(\text{variable is NaN})$) y usar modelos basados en árboles (XGBoost, LightGBM, Random Forest) o modelos de selección de Heckman / modelos de mezcla de patrones (*pattern-mixture models*).

**Ejemplos:**
1. En encuestas de compensación, ejecutivos con salarios extremadamente altos tienden a omitir la pregunta de salario.
2. En ensayos clínicos sobre depresión, pacientes con síntomas más severos abandonan el estudio (*dropout* por agravamiento del cuadro).

In [ ]:
# Ejemplo MNAR: Simulación
data_mnar = data.copy()
# Mayor probabilidad de ser NaN si el ingreso es alto
prob = (data_mnar['ingreso'] > 60000).astype(float) * 0.5 + 0.05
mask = np.random.rand(len(data_mnar)) < prob
data_mnar.loc[mask, 'ingreso'] = np.nan
print('Datos con valores faltantes MNAR:')
print(data_mnar.head(10))

#### Cuadro Comparativo de Mecanismos de Pérdida

| Criterio | MCAR (*Missing Completely at Random*) | MAR (*Missing at Random*) | MNAR (*Missing Not at Random*) |
|---|:---:|:---:|:---:|
| **Dependencia de observados** | No | Sí | Puede o no |
| **Dependencia de no observados** | No | No | **Sí** |
| **Sesgo en Listwise Deletion** | **Insesgado** (solo pierde $N$) | **Sesgado** | **Severamente Sesgado** |
| **Sesgo en Media / Mediana** | Afecta varianza / Insesga media | Sesgado | Sesgado |
| **Validez de MICE / KNN** | Válido | **Ideal y formalmente insesgado** | Parcialmente sesgado |
| **Técnica recomendada en Feature Engineering** | Imputación univariada / MICE / KNN | KNNImputer / MICE / MissForest | **Missing Indicator + Imputación + Modelos de Árboles** |
| **¿Test estadístico formal?** | Little's Test, t-test entre grupos | Correlaciones con $M$ | No directamente testeable sin datos externos |

##  Estrategias de Eliminación y Filtrado

La eliminación es el enfoque más directo. Existen tres variantes principales:

1. **Análisis de Casos Completos (*Listwise Deletion*):** Elimina cualquier fila (observación) que contenga al menos un valor faltante (`df.dropna()`).
   - **Ventaja:** Sencillo de implementar; compatible con cualquier algoritmo.
   - **Desventaja:** Pérdida masiva de datos y sesgo garantizado si el mecanismo es MAR o MNAR.
2. **Eliminación por Pares (*Pairwise Deletion*):** Utiliza todas las observaciones disponibles para cada par de variables al calcular covarianzas o correlaciones.
   - **Desventaja:** Las matrices de covarianza resultantes pueden no ser semidefinidas positivas, provocando errores en regresiones o análisis multivariados (PCA).
3. **Filtrado por Umbral de Nulidad (*Threshold Filtering*):**
   - Eliminar columnas (variables) con una tasa de faltantes superior a un umbral $\tau_{\text{col}}$ (ej. $> 60\%-70\%$), a menos que la ausencia sea una variable de negocio crítica.
   - Eliminar filas (registros) con una tasa de faltantes superior a $\tau_{\text{row}}$ (ej. $> 50\%$).
   - En pandas se utiliza el parámetro `thresh`: `df.dropna(thresh=min_datos_no_nulos)`.

In [ ]:
dfcopy.head() #dataset indicando valores faltantes

In [ ]:
#valores faltantes
dfcopy.sum()

In [ ]:
np.where(dfcopy.sum()>263)

In [ ]:
np.where(dfcopy.sum(axis=1)>60)

In [ ]:
# Filtrado por umbral (Threshold Deletion) con pandas idiomatico
# Regla: conservar columnas con al menos el 40% de datos observados (eliminar columnas con >60% NaN)
# y conservar filas con al menos el 50% de columnas completas

umbral_col = int(0.40 * len(df))
df_filtrado_cols = df.dropna(axis=1, thresh=umbral_col)

umbral_filas = int(0.50 * df_filtrado_cols.shape[1])
df_filtrado_final = df_filtrado_cols.dropna(axis=0, thresh=umbral_filas)

print(f"Dimensiones originales: {df.shape}")
print(f"Dimensiones tras filtrado de columnas (>60% NaN): {df_filtrado_cols.shape}")
print(f"Dimensiones tras filtrado final de filas (>50% NaN): {df_filtrado_final.shape}")

In [ ]:
#Eliminación de datos por registros y variables
# Simulando un Dataset con valores faltantes
df_data=pd.DataFrame(np.random.randn(100,4)+10*np.random.rand(4),columns=['A','B','C','D'])
for c in df_data.columns[:-1]:
    inan=np.random.randint(100,size=np.random.randint(20))
    df_data.loc[inan, c]=np.nan

In [ ]:
df_data

In [ ]:
# Eliminación de filas (Observaciones)
df_data.dropna()

In [ ]:
# Eliminación de columnas (Variables)
df_data.dropna(axis=1)

##  Métodos de Imputación Univariada

La imputación reemplaza cada valor faltante por una estimación plausible basada en la información observada de la **misma columna** (univariada) o de **múltiples columnas** (multivariada).

### Principios Fundamentales en Feature Engineering:
1. **Preservación de la forma de la distribución:** Los métodos univariados siempre reducen la varianza ($\sigma^2_{\text{imputado}} < \sigma^2_{\text{original}}$) y crean picos artificiales (*spikes*) en el valor imputado.
2. **Uso de Indicadores de Ausencia (*Missing Indicators*):** Para compensar la distorsión y capturar el mecanismo de pérdida, es una excelente práctica acompañar la imputación con una columna binaria $I(X = \text{NaN})$.
3. **Ajuste estricto en entrenamiento (`fit_transform` en Train, `transform` en Test):** Los valores de tendencia central calculados en entrenamiento deben aplicarse sin recalcular en el conjunto de prueba para evitar **Data Leakage**.

### 1. Imputación por Métricas de Tendencia Central

#### A. Imputación por la Media

Reemplaza cada valor faltante por la media aritmética de los valores observados:

$$\hat{x}_{i} = \bar{x}_{\text{obs}} = \frac{1}{N_{\text{obs}}} \sum_{j \in \text{Obs}} x_j$$

- **Ventajas:** Extremadamente rápida, preserva la media muestral bajo MCAR.
- **Desventajas:**
  - Solo aplicable a variables cuantitativas continuas simétricas (sensible a *outliers*).
  - Reduce la varianza muestral: $\text{Var}(X_{\text{imp}}) = \frac{N_{\text{obs}}-1}{N-1} \text{Var}(X_{\text{obs}})$.
  - Distorsiona correlaciones y covarianzas con otras variables.

In [ ]:
# Crear un DataFrame de ejemplo
data = {
    'A': [1, 2, None, 4],
    'B': [5, None, None, 8],
    'C': [9, 10, 11, 12]
}
df = pd.DataFrame(data)
print('DataFrame original:')
print(df)

In [ ]:
df.mean(numeric_only=True)

In [ ]:
# Imputar con la media
df = df.fillna(df.mean(numeric_only=True))
df

In [ ]:
from sklearn.impute import SimpleImputer

# Imputar con la media
imp_mean = SimpleImputer(strategy='mean')
imp_mean.fit_transform(df)

In [ ]:
pd.DataFrame(imp_mean.fit_transform(df), columns = df.columns)

In [ ]:
# Demostración visual de la reducción de varianza provocada por la imputación de la media
np.random.seed(42)
x_real = np.random.normal(loc=50, scale=10, size=1000)
x_con_nan = x_real.copy()
# 25% de valores faltantes
x_con_nan[np.random.rand(1000) < 0.25] = np.nan

x_imputado_media = pd.Series(x_con_nan).fillna(np.nanmean(x_con_nan))

plt.figure(figsize=(10, 4))
sns.kdeplot(x_real, label=f'Original (Var = {np.var(x_real):.2f})', color='darkblue', linewidth=2)
sns.kdeplot(x_imputado_media, label=f'Imputado Media (Var = {np.var(x_imputado_media):.2f})', color='crimson', linestyle='--', linewidth=2)
plt.axvline(np.nanmean(x_con_nan), color='crimson', linestyle=':', label='Media imputada')
plt.title('Efecto de la Imputación por la Media: Concentración de Densidad y Pérdida de Varianza', fontsize=12)
plt.xlabel('Valor de la Variable')
plt.ylabel('Densidad')
plt.legend()
plt.show()

#### B. Técnica Clave de Feature Engineering: Indicador de Ausencia (*Missing Indicator*)

Cuando imputamos un valor central (media o mediana), el modelo pierde la información de si el dato era originalmente ausente. Al añadir una **columna indicadora binaria** ($1$ si era faltante, $0$ si estaba presente), el modelo (especialmente regresión logística, SVM o redes neuronales) puede aprender un intercepto o peso separado para los registros imputados:

$$\text{Feature}_{\text{imputada}} = \text{impute}(X), \qquad \text{Feature}_{\text{is\_missing}} = I(X \text{ es NaN})$$

En `scikit-learn` se activa directamente con `add_indicator=True` dentro de `SimpleImputer`, `KNNImputer` o `IterativeImputer`.

In [ ]:
# Uso de SimpleImputer con indicador binario de ausencia integrado
from sklearn.impute import SimpleImputer

imputer_con_indicador = SimpleImputer(strategy='median', add_indicator=True)
df_ejemplo = pd.DataFrame({
    'edad': [25, np.nan, 30, 45, np.nan, 50],
    'salario': [35000, 42000, np.nan, 80000, 95000, 110000]
})

matriz_imputada = imputer_con_indicador.fit_transform(df_ejemplo)
nombres_columnas = list(df_ejemplo.columns) + [f"{col}_is_missing" for col in df_ejemplo.columns]
df_resultado_indicador = pd.DataFrame(matriz_imputada, columns=nombres_columnas)
print("Dataset con imputación de mediana + Missing Indicators:")
df_resultado_indicador

#### C. Imputación por la Mediana

Reemplaza los valores faltantes por el percentil 50 de la muestra observada:

$$\hat{x}_i = \text{Mediana}(X_{\text{obs}})$$

- **Ventaja:** Muy robusta frente a datos atípicos (*outliers*) y distribuciones asimétricas (sesgadas a la derecha o izquierda, como ingresos, precios o tiempos de espera).
- **Desventaja:** Al igual que la media, contrae artificialmente la varianza y altera las correlaciones multivariadas. Es la opción univariada por defecto preferida sobre la media en la práctica.

In [ ]:
# Crear un DataFrame de ejemplo
data = {
    'A': [1, 2, None, 4],
    'B': [5, None, None, 8],
    'C': [9, 10, 11, 12]
}
df = pd.DataFrame(data)
print('DataFrame original:')
print(df)


In [ ]:
# Imputar con la mediana
df_mediana = df.copy()
df_mediana = df_mediana.fillna(df.median(numeric_only=True))
df_mediana

In [ ]:
# Imputar con la mediana con la librería
# Imputar con la media
imp_median = SimpleImputer(strategy='median')
imp_median.fit_transform(df)

In [ ]:
# Comparación gráfica: Imputación Media vs Mediana en presencia de asimetría y outliers
np.random.seed(42)
ingresos = np.random.exponential(scale=20000, size=800) + 10000  # Asimétrico a la derecha
ingresos_nan = ingresos.copy()
ingresos_nan[np.random.rand(800) < 0.20] = np.nan

imp_mean_vals = pd.Series(ingresos_nan).fillna(np.nanmean(ingresos_nan))
imp_median_vals = pd.Series(ingresos_nan).fillna(np.nanmedian(ingresos_nan))

plt.figure(figsize=(10, 4))
sns.kdeplot(ingresos, label='Original (Distribución asimétrica)', color='black', linewidth=2)
sns.kdeplot(imp_mean_vals, label=f'Imputado Media ({np.nanmean(ingresos_nan):,.0f})', color='darkorange', linestyle='--')
sns.kdeplot(imp_median_vals, label=f'Imputado Mediana ({np.nanmedian(ingresos_nan):,.0f})', color='green', linestyle='-.')
plt.title('Comparación: Media vs Mediana en Distribución Asimétrica', fontsize=12)
plt.xlabel('Ingreso ($)')
plt.ylabel('Densidad')
plt.legend()
plt.show()

#### D. Imputación para Variables Categóricas: Moda vs. Categoría "Faltante / Desconocido"

Para variables cualitativas existen dos enfoques principales:

1. **Moda (Categoría más frecuente):**
   - Reemplaza el faltante por el valor que más se repite: $\hat{c} = \text{argmax}_{c} \text{conteo}(c)$.
   - **Recomendado:** Cuando la moda es abrumadoramente dominante ($> 80\%$) y la tasa de faltantes es muy baja ($< 5\%$).
   - **Riesgo:** Si la moda no es dominante, sobre-representa la categoría modal y desbalancea las frecuencias.

2. **Creación de una Categoría Explícita (`'Desconocido'` / `'Missing'`):**
   - Trata la ausencia como una nueva categoría válida en lugar de adivinar una categoría existente.
   - **Especialmente útil cuando el mecanismo es MNAR** o cuando la falta de respuesta tiene significado propio.

In [ ]:
# Imputar con la moda
df_moda = df.copy()
moda = df.mode().iloc[0]
df_moda = df_moda.fillna(moda)
df_moda

In [ ]:
# Imputar con la moda
imp_mode = SimpleImputer(strategy='most_frequent')
imp_mode.fit_transform(df)

##### Ejemplo

In [ ]:
# Imputación por categoría constante ('Desconocido' o 'Missing')
imp_constant = SimpleImputer(strategy='constant', fill_value='Desconocido')
df_cat_ejemplo = pd.DataFrame({
    'ciudad': ['Guadalajara', 'CDMX', np.nan, 'Monterrey', np.nan, 'Guadalajara']
})

df_cat_imputado = pd.DataFrame(imp_constant.fit_transform(df_cat_ejemplo), columns=['ciudad'])
print("Frecuencias tras imputar como categoría explícita 'Desconocido':")
print(df_cat_imputado['ciudad'].value_counts())

In [ ]:
# otro ejemplo titanic
# Seleccionar columnas categóricas con valores faltantes
df_mo_titanic = titanic_df
df_mo_titanic

In [ ]:
df_mo_titanic.select_dtypes('object').columns

In [ ]:
df_mo_titanic.select_dtypes('object').isnull().sum()

In [ ]:
titanic_cat_mode = SimpleImputer(strategy='most_frequent')
titanic_cat_mode.fit_transform(df_mo_titanic.select_dtypes('object'))

#### E. Imputación por Valor Arbitrario o Extremo

Consiste en reemplazar los valores faltantes por un número que no se encuentra en el rango natural de la variable (por ejemplo, $-999$, $-1$ o el percentil $99 + 3 \times \text{IQR}$).

- **Ventajas:** Muy eficaz para algoritmos basados en árboles de decisión (Decision Tree, Random Forest, XGBoost), ya que el árbol puede crear una partición limpia que aísle los datos faltantes en una rama propia.
- **Desventaja:** **Peligroso para modelos lineales o basados en distancia** (KNN, SVM, Regresión Lineal/Logística, Redes Neuronales), donde un valor extremo distorsiona fuertemente los gradientes y coeficientes.

In [ ]:
# Imputación por valor arbitrario extremo (-999) para modelos de árboles
imp_arbitrario = SimpleImputer(strategy='constant', fill_value=-999)
df_num_arb = pd.DataFrame({'edad': [23, np.nan, 45, np.nan, 60]})
df_arb_res = pd.DataFrame(imp_arbitrario.fit_transform(df_num_arb), columns=['edad'])
print("Dataset imputado con valor extremo (-999):")
print(df_arb_res)

## Imputación en Series Temporales y Datos Secuenciales

Cuando las observaciones poseen un **orden cronológico o secuencial intrínseco**, la media global no tiene sentido físico. Se utilizan métodos basados en la trayectoria temporal:

### 1. Forward Fill (`ffill`) y Backward Fill (`bfill`)
- **Forward Fill (Última Observación Llevada Hacia Adelante - LOCF):** $x_t = x_{t-1}^*$. Asume que el estado del sistema persiste hasta que ocurra un nuevo evento.
- **Backward Fill (Próxima Observación Llevada Hacia Atrás - NOCB):** $x_t = x_{t+1}^*$.
- **Combinación robusta:** `df.ffill().bfill()` para asegurar que no queden nulos al inicio ni al final de la serie.

### 2. Interpolación Matemática (Lineal, Polinomial, Spline)
Calcula el valor faltante asumiendo una curva suave o recta entre los puntos conocidos adyacentes:

$$\hat{x}_t = x_{t_0} + \frac{x_{t_1} - x_{t_0}}{t_1 - t_0}(t - t_0)$$

En `pandas` se ejecuta con `.interpolate(method='linear')`, `.interpolate(method='quadratic')` o `.interpolate(method='time')`.

In [ ]:
# Crear una serie temporal con valores faltantes
np.random.seed(0)
dates = pd.date_range('2023-01-01', periods=20)
values = np.random.randn(20).cumsum()
series = pd.Series(values, index=dates)

# Introducir valores faltantes
series.iloc[[3, 4, 10, 11, 12, 17]] = np.nan
print('Serie original con valores faltantes:')
print(series)


In [ ]:
# Comparativa de técnicas temporales: ffill, bfill e interpolación lineal y spline
series_ffill = series.ffill()
series_bfill = series.bfill()
series_interp_lin = series.interpolate(method='linear')
series_interp_spline = series.interpolate(method='spline', order=2)

In [ ]:
series_ffill

In [ ]:
# Visualización comparativa de métodos temporales
plt.figure(figsize=(12, 6))
plt.plot(series.index, series.values, 'ko', markersize=8, label='Datos Observados (Con NAs)', zorder=5)
plt.plot(series_ffill.index, series_ffill.values, 's--', color='orange', alpha=0.8, label='Forward Fill (LOCF)')
plt.plot(series_bfill.index, series_bfill.values, 'd--', color='purple', alpha=0.6, label='Backward Fill (NOCB)')
plt.plot(series_interp_lin.index, series_interp_lin.values, 'g^-', linewidth=2, label='Interpolación Lineal')
plt.plot(series_interp_spline.index, series_interp_spline.values, 'r.-', linewidth=2, label='Interpolación Spline (Orden 2)')

plt.xticks(rotation=30, ha='right')
plt.grid(True, linestyle=':', alpha=0.6)
plt.legend(loc='best', frameon=True)
plt.title('Comparación de Métodos de Imputación para Series Temporales', fontsize=13)
plt.xlabel('Fecha')
plt.ylabel('Valor de la Serie')
plt.tight_layout()
plt.show()

- Es recomendable usar estos métodos solo cuando los datos tienen un orden natural (por ejemplo, tiempo).
- Si los valores faltantes están al inicio o final de la serie, forward fill o backward fill pueden no imputar todos los valores.
- Se pueden combinar ambos métodos para imputar valores al inicio y final:

```python
serie_imputada = serie.ffill().bfill()
```

## Imputación por Donantes (Hot Deck y Cold Deck)

Los métodos de donantes sustituyen los valores faltantes por valores reales observados de otros registros ("donantes").

### 1. Imputación Hot Deck
El donante proviene del **mismo conjunto de datos**.
- **Aleatorio simple:** Se extrae un valor al azar de la distribución observada ($x_{i,\text{imp}} = x_{j,\text{obs}}$). Preserva la distribución empírica original y la varianza, a diferencia de la media.
- **Estratificado / Por similitud:** Se divide el dataset en grupos/estratos (ej. por edad o género) y se selecciona un donante dentro del mismo grupo.

### 2. Imputación Cold Deck
El donante proviene de una **fuente externa o histórica** (ej. censo anterior, tabla estándar del sector).

$$x_{i,\text{imp}} = x_{k,\text{externo}}, \quad k \in \mathcal{D}_{\text{externo}}$$

In [ ]:
# Simulación de datos
np.random.seed(42)
df = pd.DataFrame({
    'edad': np.random.randint(18, 65, 20),
    'ingreso': np.random.randint(10000, 50000, 20)
})
# Introducir valores faltantes
df.loc[[2, 5, 7, 12], 'ingreso'] = np.nan
print('Datos originales con valores faltantes:')
print(df)

#### A. Hot Deck Aleatorio Simple
Preserva la varianza empírica muestreando con reemplazo de los datos observados:

In [ ]:
# Imputación Hot Deck aleatoria
df_imp_hot_deck=df.copy()
donantes = df['ingreso'].dropna()
for idx in df_imp_hot_deck[df_imp_hot_deck['ingreso'].isnull()].index:
    df_imp_hot_deck.loc[idx,'ingreso'] = np.random.choice(donantes)
df_imp_hot_deck

In [ ]:
#grupo edad
df['grupo_edad'] = pd.cut(df['edad'], bins=[17, 30, 45, 65], labels=['Joven', 'Adulto', 'Mayor'])
df.head()

In [ ]:
# Introducimos de nuevo valores faltantes para el ejemplo
df.loc[[2, 5, 7, 12], 'ingreso'] = np.nan
df_imp_hotdeck_group = df.copy()
df_imp_hotdeck_group.head(15)

#### B. Hot Deck Estratificado por Grupo
El donante se selecciona dentro del mismo estrato (por ejemplo, mismo grupo de edad):

In [ ]:
# Imputación Hot Deck por grupo
for idx, row in df_imp_hotdeck_group[df_imp_hotdeck_group['ingreso'].isnull()].iterrows():
    grupo = row['grupo_edad']
    donantes = df_imp_hotdeck_group[(df_imp_hotdeck_group['grupo_edad']==grupo) & (df_imp_hotdeck_group['ingreso'].notnull())]['ingreso']
    if not donantes.empty:
        df_imp_hotdeck_group.loc[idx, 'ingreso'] = np.random.choice(donantes)
    else:
        df_imp_hotdeck_group.loc[idx, 'ingreso'] = np.random.choice(df_imp_hotdeck_group['ingreso'].dropna())
df_imp_hotdeck_group
 

In [ ]:
#Imputación Cold Deck usando un Dataset Externo: Supongamos que tenemos un dataset histórico con la misma estructura y lo usamos como fuente de donantes
#Dataset externo (histórico)
df_ext = pd.DataFrame({
    'edad': np.random.randint(18, 65, 20),
    'ingreso': np.random.randint(12000, 48000, 20)
})
df_ext

In [ ]:
# Imputación Cold Deck: tomar valores de ingreso del dataset externo
df_imp_cold_deck=df.copy()
donantes = df_ext['ingreso'].dropna()
for idx in df_imp_cold_deck[df_imp_cold_deck['ingreso'].isnull()].index:
    df_imp_cold_deck.loc[idx,'ingreso'] = np.random.choice(donantes)
df_imp_cold_deck

## Imputación Multivariada por Modelos Predictivos

A diferencia de los métodos univariados, la **imputación multivariada** aprovecha la matriz de correlaciones y dependencias entre todas las variables observadas para predecir el valor faltante. Es especialmente poderosa bajo el mecanismo **MAR**.

```
  Variables Observadas (X1, X2, ..., Xp)  ──► [ MODELO PREDICTIVO ] ──► Estimación de Y_faltante
```

### 1. Imputación por Regresión Lineal Determinística

Ajusta un modelo de regresión sobre las observaciones completas y evalúa la ecuación en las filas con valores faltantes:

$$\hat{y}_i = \hat{\beta}_0 + \hat{\beta}_1 x_{i1} + \hat{\beta}_2 x_{i2} + \dots + \hat{\beta}_p x_{ip}$$

- **Problema de la Regresión Determinística:** Todos los puntos imputados caen **exactamente sobre el hiperplano de regresión** ($\text{error} = 0$). Esto **sobreestima la fuerza de la correlación** ($r=1$ condicional) y subestima la variabilidad del error residual.

In [ ]:
#Supongamos un dataset con varias variables predictoras. Imputaremos valores faltantes en 'y' usando regresión múltiple.
from sklearn.linear_model import LinearRegression

# Simulación de datos con múltiples variables
np.random.seed(42)
n = 150
X1 = np.random.normal(5, 2, n)
X2 = np.random.normal(10, 3, n)
X3 = np.random.normal(20, 5, n)
y = 3 + 2*X1 - 1.5*X2 + 0.5*X3 + np.random.normal(0, 2, n)
df_multi = pd.DataFrame({'X1': X1, 'X2': X2, 'X3': X3, 'y': y})
df_multi.head()

In [ ]:
# Introducimos valores faltantes en 'y' (simulando valores faltantes)
mask = np.random.rand(n) < 0.18
df_multi.loc[mask, 'y'] = np.nan
df_multi.head()

In [ ]:
df_multi.isnull().sum()

In [ ]:
# Separar datos completos e incompletos
df_complete = df_multi[df_multi['y'].notnull()]
df_complete.isnull().sum()

In [ ]:
df_missing = df_multi[df_multi['y'].isnull()]
df_missing.isnull().sum()

In [ ]:
# Ajustar regresión múltiple
model = LinearRegression()
model.fit(df_complete[['X1','X2','X3']], df_complete['y'])


In [ ]:
#\hat{y}
model.predict(df_complete[['X1','X2','X3']])

In [ ]:
y_faltantes = model.predict(df_missing[['X1','X2','X3']])
y_faltantes

In [ ]:
# Imputar valores faltantes
df_multi.loc[df_multi['y'].isnull(), 'y'] = y_faltantes
df_multi

In [ ]:
df_multi.isnull().sum()

**Desventajas:**
- Si se usan modelos paramétricos de regresión se pueden llegar a producir estimaciones sesgadas. Por ejemplo, si se usa un modelo lineal los valores imputados caerán en una línea recta o en el hiperplano, dependiendo de la cantidad de dimensiones.
- La correlación entre los datos imputados es igual a 1, por lo que las correlaciones estarían sobreestimadas.


### 2. Imputación por Regresión Estocástica

Para restaurar la dispersión y la incertidumbre natural de los datos, la **regresión estocástica** añade un término de perturbación aleatorio muestreado de los residuos del modelo o de una distribución normal $\mathcal{N}(0, \hat{\sigma}^2_\varepsilon)$:

$$\hat{y}_{i,\text{estocástico}} = \hat{\beta}_0 + \sum_{j=1}^p \hat{\beta}_j x_{ij} + e_i^*, \qquad e_i^* \sim \text{Residuos}(y_{\text{obs}} - \hat{y}_{\text{obs}})$$

**Ventaja:** Preserva tanto las correlaciones entre variables como la varianza residual, siendo la base fundamental de los métodos MICE.

In [ ]:
# Simulación de datos
np.random.seed(123)
n = 120
X1 = np.random.normal(8, 2, n)
X2 = np.random.normal(15, 4, n)
y = 5 + 1.2*X1 - 0.8*X2 + np.random.normal(0, 3, n)
df_stoch = pd.DataFrame({'X1': X1, 'X2': X2, 'y': y})

# Introducimos valores faltantes en 'y'
mask = np.random.rand(n) < 0.2
df_stoch.loc[mask, 'y'] = np.nan
df_stoch.head()

In [ ]:
# Separar datos completos e incompletos
df_complete = df_stoch[df_stoch['y'].notnull()]
df_missing = df_stoch[df_stoch['y'].isnull()]


In [ ]:
# Ajustar modelo de regresión
model = LinearRegression()
model.fit(df_complete[['X1', 'X2']], df_complete['y'])

# Calcular residuos del modelo
residuals = df_complete['y'] - model.predict(df_complete[['X1', 'X2']])
residuals

In [ ]:
# Imputar valores faltantes añadiendo ruido aleatorio (regresión estocástica)
imputed_values = model.predict(df_missing[['X1', 'X2']]) + np.random.choice(residuals, size=len(df_missing))
df_stoch.loc[df_stoch['y'].isnull(), 'y'] = imputed_values
df_stoch.head()

###  Imputación de Valores Faltantes por KNN (K-Nearest Neighbors)

El método $KNN$ es un modelo predictivo cuyas aplicaciones incluyen la imputación de datos, ya que es un clasificador de aprendizaje supervisado no paramétrico, que utiliza la proximidad para hacer clasificaciones o predicciones sobre la agrupación de un punto de datos individual. En otras palabras, **se estima el valor perdido como la media (en el caso de las variables numéricas) de los valores de los $k$ vecinos u observaciones más cercanos. Así mismo, para las variables categóricas, se utiliza la clase mayoritaria de entre los k más cercanos.**

El valor de $k$ define cuántos vecinos se verificarán para determinar la clasificación del dato faltante, siendo k directamente proporcional a la generación de sesgo e inversamente proporcional a la varianza. En general se recomienda tener un número impar de k para evitar empates en la clasificación.

Para cada valor faltante, el algoritmo:
1. Calcula la distancia entre la observación incompleta y todas las observaciones completas (usualmente distancia euclidiana).
2. Selecciona los k vecinos más cercanos.
3. Imputa el valor faltante usando la media (para variables numéricas) o la moda (para categóricas) de los vecinos.


**Distancia Euclidiana:**
$$
d(x, y) = \sqrt{\sum_{i=1}^n (x_i - y_i)^2}
$$

**Imputación numérica:**
$$
\hat{x}_{\text{miss}} = \frac{1}{k} \sum_{j=1}^k x_{j, \text{vecino}}
$$

**Imputación categórica:**
- Se utiliza la moda de los vecinos.

Algunas ventajas de este método son: aprovecha la similitud entre observaciones y puede preservar relaciones complejas entre variables.

**Desventajas:** Computacionalmente costoso para grandes datasets. Sensible a la escala de las variables (es recomendable normalizar antes de imputar). El valor de $k$ debe seleccionarse cuidadosamente.

In [ ]:
## Imputación para variables cuantitativas
from sklearn.datasets import load_iris
from sklearn.preprocessing import StandardScaler
from sklearn.impute import KNNImputer
# Cargar datos
iris = load_iris(as_frame=True)
df_iris = iris.data.copy()

# Simular valores faltantes
np.random.seed(1)
mask = np.random.rand(*df_iris.shape) < 0.1
df_iris[mask] = np.nan

# Normalizar antes de imputar
scaler = StandardScaler()
df_scaled = pd.DataFrame(scaler.fit_transform(df_iris), columns=df_iris.columns)

In [ ]:
# Imputación KNN
imputer = KNNImputer(n_neighbors=5)
imputer.fit_transform(df_scaled)

In [ ]:
df_imputed = pd.DataFrame(imputer.fit_transform(df_scaled), columns = df_scaled.columns)

In [ ]:
df_final = pd.DataFrame(scaler.inverse_transform(df_imputed), columns = df_scaled.columns)
df_final

In [ ]:
df_final.isnull().sum()

In [ ]:
## Imputación para variables Categóricas
# Para variables categóricas, se recomienda codificar las categorías numéricamente antes de imputar y luego decodificar tras la imputación.
df_cat = pd.DataFrame({
    'color': ['rojo', 'azul', 'verde', np.nan, 'azul', 'rojo', np.nan],
    'forma': ['circulo', 'cuadro', 'triangulo', 'cuadro', np.nan, 'circulo', 'triangulo']
})
df_cat

#### Imputación KNN para Variables Categóricas
Para aplicar KNN a variables categóricas:
1. Codificar las categorías en valores numéricos enteros o mediante *One-Hot Encoding*.
2. Mantener explícitamente los valores `NaN` (no codificarlos como un número válido como $-1$).
3. Aplicar `KNNImputer`.
4. Redondear y decodificar de regreso a las etiquetas categóricas originales.

In [ ]:
# Codificar categorías
df_cat_trans = df_cat.copy()
for col in df_cat_trans.columns:
    df_cat_trans[col] = df_cat_trans[col].astype('category').cat.codes.replace(-1, np.nan)
df_cat_trans


In [ ]:
# Imputar con KNN
imputer = KNNImputer(n_neighbors=2)
df_cat_imp = pd.DataFrame(imputer.fit_transform(df_cat_trans), columns = df_cat_trans.columns)
df_cat_imp

In [ ]:
# Decodificar
for col in df_cat.columns:
    cats = ['azul', 'circulo', 'cuadro', 'rojo', 'triangulo', 'verde']
    # Ajustar categorías según columna
    if col == 'color':
        mapping = {0: 'azul', 1: 'rojo', 2: 'verde'}
    else:
        mapping = {0: 'circulo', 1: 'cuadro', 2: 'triangulo'}
    df_cat_imp[col] = df_cat_imp[col].round().astype(int).map(mapping)

print('Datos categóricos tras imputación KNN:')
df_cat_imp

In [ ]:
df_cat

### Imputación de Valores Faltantes con Missing Forest (MissForest)

La imputación por "Missing Forest" (MissForest) es un método basado en bosques aleatorios (Random Forests) para imputar valores faltantes en datasets tanto numéricos como categóricos. Es un método no paramétrico y robusto que puede capturar relaciones no lineales y complejas entre variables.

**MissForest** utiliza un enfoque iterativo:
1. Inicializa los valores faltantes (por ejemplo, con la media o moda).
2. Para cada variable con valores faltantes, entrena un Random Forest usando las otras variables como predictores.
3. Imputa los valores faltantes usando las predicciones del modelo.
4. Repite el proceso para todas las variables con valores faltantes hasta que la imputación converge o se alcanza un número máximo de iteraciones.

Para una variable $X_j$ con valores faltantes:

$$X_{j,\text{miss}} = f_{RF}(X_{-j,\text{obs}})$$

donde $f_{RF}$ es el modelo de Random Forest ajustado usando las otras variables $X_{-j}$ como predictores y los valores observados de $X_j$ como objetivo.

El proceso se repite para cada variable con valores faltantes y se actualizan las imputaciones en cada iteración.

Puede manejar variables numéricas y categóricas. Captura relaciones no lineales y complejas. No requiere supuestos de distribución.

**Desventajas:** Computacionalmente intensivo para grandes datasets. Puede sobreajustar si hay pocos datos o muchas variables irrelevantes.

In [ ]:
# Imputación tipo MissForest manual para variables numéricas usando scikit-learn
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor

# Simulación de datos
np.random.seed(0)
df = pd.DataFrame({
    'A': np.random.normal(10, 2, 20),
    'B': np.random.normal(5, 1, 20)
})
df.loc[[2, 5, 7], 'A'] = np.nan
df.loc[[1, 6, 12], 'B'] = np.nan

print('Datos originales con valores faltantes:')
df.head()


In [ ]:
# Imputación iterativa tipo MissForest (solo numérico, ejemplo básico)
df_imputed = df.copy()
for col in df.columns:
    mask = df_imputed[col].isnull()
    if mask.any():
        train = df_imputed.loc[~mask]
        test = df_imputed.loc[mask]
        X_train = train.drop(columns=[col])
        y_train = train[col]
        X_test = test.drop(columns=[col])
        # Imputar valores faltantes en predictores con la media temporalmente
        X_train = X_train.fillna(X_train.mean())
        X_test = X_test.fillna(X_train.mean())
        rf = RandomForestRegressor(n_estimators=100, random_state=0)
        rf.fit(X_train, y_train)
        df_imputed.loc[mask, col] = rf.predict(X_test)

print('\nDatos tras imputación tipo MissForest manual:')
df_imputed.head()

### Imputación múltiple (MICE)
La imputación múltiple se caracteriza por devolver más de un valor para cada valor faltante. Cada uno de los valores faltantes se imputan m veces, obteniendo m conjuntos de datos completos. Estos múltiples valores se combinan para obtener los valores imputados. Para combinar estos valores se puede usar la media o mediana en el caso de variables numéricas, mientras que para variables categóricas podemos emplear la moda. También se podría escoger
uno de los valores de forma aleatoria.

El proceso de imputación múltiple típicamente sigue estos pasos:
1. **Imputación:** Se generan $m$ datasets completos, imputando los valores faltantes de manera diferente en cada uno (usando métodos estocásticos).
2. **Análisis:** Se realiza el análisis estadístico deseado en cada dataset imputado.
3. **Combinación:** Se combinan los resultados de los $m$ análisis para obtener estimaciones finales y errores estándar ajustados.

Sea $\hat{Q}_i$ la estimación del parámetro de interés en el dataset imputado $i$ ($i=1,\ldots,m$), y $U_i$ su varianza estimada.

- **Estimación combinada:**
$$
\bar{Q} = \frac{1}{m} \sum_{i=1}^m \hat{Q}_i
$$

- **Varianza total:**
$$
T = \bar{U} + \left(1 + \frac{1}{m}\right)B
$$
donde:
$$
\bar{U} = \frac{1}{m} \sum_{i=1}^m U_i \quad \text{y} \quad B = \frac{1}{m-1} \sum_{i=1}^m (\hat{Q}_i - \bar{Q})^2
$$

Al generar varias imputaciones por cada valor faltante y combinarlas, estamos realizando estimaciones más precisas y menos sesgadas de los valores faltantes. Otra ventaja es su posible aplicación tanto en variables numéricas como variables categóricas. Sus desventajas podrían ser el coste computacional de realizar varias imputaciones para cada valor faltante y el hecho de elegir un criterio para combinar estos valores.



In [ ]:
from sklearn.experimental import enable_iterative_imputer  # noqa
from sklearn.impute import IterativeImputer

# Simulación de datos con valores faltantes
np.random.seed(0)
df = pd.DataFrame({
    'A': np.random.normal(10, 2, 20),
    'B': np.random.normal(5, 1, 20),
    'C': np.random.normal(0, 1, 20)
})
df.loc[[2, 5, 7], 'A'] = np.nan
df.loc[[1, 6, 12], 'B'] = np.nan
df.loc[[3, 8, 13], 'C'] = np.nan

print('Datos originales con valores faltantes:')
df


In [ ]:
# Imputación múltiple (MICE) - se puede repetir varias veces para obtener diferentes imputaciones
imputer = IterativeImputer(random_state=0, sample_posterior=True, max_iter=10)
imputed_datasets = []
for i in range(5):
    imputer.random_state=i
    imputed = pd.DataFrame(imputer.fit_transform(df), columns=df.columns)
    imputed_datasets.append(imputed)
imputed

In [ ]:
imputed_datasets[0]

In [ ]:
imputed_datasets

In [ ]:
imputed_datasets[np.random.choice(range(len(imputed_datasets)))]

In [ ]:
# Calcular la media y varianza de 'A' en cada dataset imputado
means = np.array([d['A'].mean() for d in imputed_datasets])
vars_ = np.array([d['A'].var(ddof=1)/len(d) for d in imputed_datasets])  # varianza de la media

In [ ]:
# Estimación combinada y varianza total
Q_bar = means.mean()
U_bar = vars_.mean()
B = means.var(ddof=1)
T = U_bar + (1 + 1/len(means)) * B
std_error = np.sqrt(T)

print(f"Media combinada de 'A': {Q_bar:.3f}")
print(f"Error estándar combinado: {std_error:.3f}")

In [ ]:
#OTRO EJEMPLO
from sklearn.datasets import load_iris

# Cargar datos
iris = load_iris(as_frame=True)
df_iris = iris.data.copy()

# Simular valores faltantes
np.random.seed(1)
mask = np.random.rand(*df_iris.shape) < 0.1
df_iris[mask] = np.nan
df_iris.head()

In [ ]:
# Imputación múltiple (MICE)
imputer = IterativeImputer(random_state=0, sample_posterior=True, max_iter=10)
imputed_iris = []
for i in range(5):
    imputer.random_state = i
    imputed = pd.DataFrame(imputer.fit_transform(df_iris), columns=df_iris.columns)
    imputed_iris.append(imputed)

print('Primeras filas de la primera imputación en Iris:')
print(imputed_iris[0].head())

In [ ]:
#Otro ejemplo
file_path = "https://raw.githubusercontent.com/selva86/datasets/master/Churn_Modelling_m.csv"
df = pd.read_csv(file_path)
df.head()

In [ ]:
df.info()

In [ ]:
df.isnull().sum()

In [ ]:
df_train = df.loc[:, ["Balance", "Age", "Exited"]] # Usamos tres características numéricas
df_train.head()

In [ ]:
imputer.fit(df_train)

In [ ]:
df_imputed = imputer.transform(df_train)

In [ ]:
df_imputed

In [ ]:
df.loc[:, ["Balance", "Age", "Exited"]] = df_imputed
df.head(10)

In [ ]:
df.isnull().sum()

## Actividad: Caso de Estudio Titanic

En este ejercicio práctico aplicaremos el flujo completo de tratamiento de datos faltantes sobre el dataset histórico del **Titanic**:
1. Diagnóstico y cuantificación de nulidad.
2. Selección de estrategias según el mecanismo y tipo de variable:
   - `deck` (>77% faltante): Tratamiento como categoría `'Desconocido'` o descarte.
   - `embarked` / `embark_town` (<0.3% faltante): Imputación por moda.
   - `age` (~20% faltante, MAR condicional a `pclass` y `fare`): Comparación de **Mediana**, **KNNImputer** y **MICE (`IterativeImputer`)**.
3. Evaluación comparativa visual de las densidades resultantes.

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import missingno as msno
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.experimental import enable_iterative_imputer  # noqa
from sklearn.impute import IterativeImputer

In [ ]:
# 1. Cargar el dataset Titanic
df_titanic = sns.load_dataset('titanic')
print(f"Dimensiones del dataset Titanic: {df_titanic.shape}")
df_titanic.head()

In [ ]:
# 2. Análisis exploratorio de valores faltantes en Titanic
reporte_titanic = pd.DataFrame({
    'Tipo': df_titanic.dtypes,
    'Faltantes': df_titanic.isna().sum(),
    'Porcentaje (%)': (df_titanic.isna().mean() * 100).round(2)
}).sort_values(by='Faltantes', ascending=False)

print("Reporte de variables con datos faltantes:")
display(reporte_titanic[reporte_titanic['Faltantes'] > 0])

In [ ]:
# Seleccionar variables relevantes para el modelado
cols_modelo = ['survived', 'pclass', 'sex', 'age', 'sibsp', 'parch', 'fare', 'embarked']
df_sub = df_titanic[cols_modelo].copy()

# Codificar 'sex' y 'embarked' numéricamente para permitir imputadores multivariados
df_sub['sex'] = df_sub['sex'].map({'male': 0, 'female': 1})
df_sub['embarked'] = df_sub['embarked'].astype('category').cat.codes.replace(-1, np.nan)

print("Primeras filas del subconjunto procesado:")
df_sub.head()

In [ ]:
# Visualizar distribución de 'age' antes de imputar
plt.figure(figsize=(8, 4))
sns.histplot(df_sub['age'].dropna(), kde=True, color='teal', bins=30)
plt.title('Distribución Original de Edad (Observados sin NaN)', fontsize=12)
plt.xlabel('Edad (años)')
plt.ylabel('Frecuencia')
plt.show()

In [ ]:
# 3. Imputación de variables numéricas

# a) Imputación con la mediana
imp_median = SimpleImputer(strategy='median')
age_imputada_mediana = imp_median.fit_transform(df_sub[['age']]).ravel()

In [ ]:
# b) Imputación con KNN (escalando previamente las variables numéricas)
from sklearn.preprocessing import MinMaxScaler

scaler_knn = MinMaxScaler()
df_scaled_knn = scaler_knn.fit_transform(df_sub)

knn_imp = KNNImputer(n_neighbors=5, weights='distance')
df_knn_imputed_scaled = knn_imp.fit_transform(df_scaled_knn)
df_knn_final = pd.DataFrame(scaler_knn.inverse_transform(df_knn_imputed_scaled), columns=df_sub.columns)

age_imputada_knn = df_knn_final['age']

In [ ]:
# c) Imputación con IterativeImputer (MICE)
mice_imp = IterativeImputer(max_iter=15, random_state=42, sample_posterior=True)
df_mice_imputed = pd.DataFrame(mice_imp.fit_transform(df_sub), columns=df_sub.columns)

age_imputada_mice = df_mice_imputed['age']

In [ ]:
# Comparación de densidades de probabilidad tras la imputación en 'age'
plt.figure(figsize=(12, 6))

sns.kdeplot(df_sub['age'].dropna(), label=f"Original sin NaN (Media: {df_sub['age'].mean():.1f}, Desv: {df_sub['age'].std():.1f})", color='black', linewidth=2.5)
sns.kdeplot(age_imputada_mediana, label=f"Mediana Univariada (Media: {age_imputada_mediana.mean():.1f}, Desv: {age_imputada_mediana.std():.1f})", color='crimson', linestyle='--', linewidth=2)
sns.kdeplot(age_imputada_knn, label=f"KNN Imputer (Media: {age_imputada_knn.mean():.1f}, Desv: {age_imputada_knn.std():.1f})", color='blue', linestyle=':', linewidth=2)
sns.kdeplot(age_imputada_mice, label=f"MICE Iterative (Media: {age_imputada_mice.mean():.1f}, Desv: {age_imputada_mice.std():.1f})", color='forestgreen', linestyle='-.', linewidth=2)

plt.title('Comparación de Distribuciones tras Distintos Métodos de Imputación (Titanic - Age)', fontsize=13)
plt.xlabel('Edad')
plt.ylabel('Densidad')
plt.legend(frameon=True, facecolor='white', framealpha=0.9)
plt.grid(True, linestyle=':', alpha=0.5)
plt.show()

Una de las reglas más críticas en la Ingeniería de Características es:

> **REGLA DE ORO:** NUNCA imputar el conjunto completo antes de realizar la partición *Train / Test*. 
> La imputación debe ajustarse (`fit`) **exclusivamente sobre el conjunto de entrenamiento** y aplicarse (`transform`) sobre el conjunto de prueba para evitar **Data Leakage**.


### Árbol de Decisión Práctico

| Escenario / Tipo de Dato | Mecanismo Probable | Estrategia Recomendada | ¿Por qué? |
|---|:---:|---|---|
| **Poco % faltantes (<3-5%), variable no crítica** | MCAR / MAR | `SimpleImputer(strategy='median')` o `'most_frequent'` | Rápido, mínimo impacto en varianza global. |
| **Variable numérica con asimetría u outliers** | MCAR / MAR | `SimpleImputer(strategy='median', add_indicator=True)` | Mediana es robusta y el indicador captura la señal de ausencia. |
| **Variable categórica con valores faltantes** | MAR / MNAR | Categoría explícita `'Desconocido'` o `SimpleImputer(strategy='most_frequent')` | Evita forzar etiquetas cuando no hay certidumbre. |
| **Series temporales / Sensores continuos** | MAR temporal | `ffill().bfill()` o `interpolate(method='linear'/'time')` | Mantiene la continuidad física y secuencial. |
| **Fuertes correlaciones lineales entre variables** | MAR | `IterativeImputer` (MICE con regresión lineal) o `KNNImputer` | Reconstruye el valor a partir de los patrones observados. |
| **Relaciones no lineales complejas / Interacciones** | MAR | `MissForest` o `IterativeImputer(estimator=ExtraTreesRegressor())` | Captura no linealidades sin supuestos distribucionales. |
| **Presunto mecanismo no aleatorio (salarios, sesgos)** | MNAR | **Missing Indicator Obligatorio** + Modelos basados en árboles (LightGBM/XGBoost) | El árbol aprende la regla de negocio asociada al valor ausente. |
